# 01 · Estandarización de FIRMS Bolivia (MODIS + VIIRS S-NPP + VIIRS NOAA-20)
Construye la **tabla maestra** con un esquema común, pensada para la armonización MODIS–VIIRS del desafío NASA Space Apps.

Principios:
- **Nada se borra.** Los filtros (tipo 0, confianza) son columnas booleanas, no eliminaciones. Así se puede analizar con y sin confianza baja.
- Se conservan las temperaturas originales de cada sensor (canales similares pero no idénticos) y la fecha UTC original.
- Los sensores **no se suman**: cada fila queda identificada por `source`.

Entradas: los tres CSV unidos (ver `unir_*.ipynb`). Salida: `DATOS_PROCESADOS/`.

In [1]:
from pathlib import Path
import pandas as pd

RAIZ = Path.cwd()
SALIDA = RAIZ / "DATOS_PROCESADOS"
SALIDA.mkdir(exist_ok=True)

ENTRADAS = {
    "MODIS": RAIZ / "MODIS" / "modis_Bolivia_2000_2025.csv",
    "VIIRS-SNPP": RAIZ / "VIIRS-S-NPP" / "viirs-snpp_Bolivia_2012_2025.csv",
    "VIIRS-NOAA20": RAIZ / "VIIRS-NOAA" / "viirs-jpss1_Bolivia_2018_2025.csv",
}
for k, p in ENTRADAS.items():
    print(k, p.exists())

MODIS True
VIIRS-SNPP True
VIIRS-NOAA20 True


## 1. Equivalencias entre sensores
| Concepto | MODIS | VIIRS |
|---|---|---|
| Temp. IR medio (K) | `brightness` (canales 21/22) | `bright_ti4` (I4) |
| Temp. IR térmico (K) | `bright_t31` (canal 31) | `bright_ti5` (I5) |
| Confianza | 0–100 | `l` / `n` / `h` |
| Satélite | `Terra`, `Aqua` | `N` = Suomi-NPP, `N20` = NOAA-20 |
| Resolución nominal | 1000 m | 375 m |

Clases de confianza MODIS: 0–29 baja, 30–79 nominal, 80–100 alta (guía de usuario MODIS). Fuente de los códigos: NASA FIRMS, *Active Fire Data Attributes for MODIS and VIIRS*.

In [2]:
PLATAFORMA = {"Terra": "Terra", "Aqua": "Aqua", "N": "Suomi-NPP", "N20": "NOAA-20"}
CLASE_VIIRS = {"l": "baja", "n": "nominal", "h": "alta"}


def clase_modis(c):
    return pd.cut(c, bins=[-1, 29, 79, 100], labels=["baja", "nominal", "alta"]).astype(str)


def estandarizar(df, sensor, temp_medio, temp_termico, archivo):
    out = pd.DataFrame()
    out["latitude"] = df["latitude"]
    out["longitude"] = df["longitude"]
    # fecha-hora UTC original + hora boliviana (UTC-4). Se guardan ambas.
    hhmm = df["acq_time"].str.zfill(4)
    utc = pd.to_datetime(df["acq_date"].astype(str) + " " + hhmm.str[:2] + ":" + hhmm.str[2:], format="%Y-%m-%d %H:%M")
    out["acq_datetime_utc"] = utc
    bo = utc - pd.Timedelta(hours=4)
    out["acq_datetime_bo"] = bo
    out["acq_date_bo"] = bo.dt.normalize()
    out["acq_hour_bo"] = bo.dt.hour.astype("int8")
    out["platform"] = df["satellite"].map(PLATAFORMA)
    out["sensor"] = sensor
    out["source"] = sensor + "-" + out["platform"]
    out["resolucion_nominal_m"] = 1000 if sensor == "MODIS" else 375
    out["confidence_original"] = df["confidence"].astype(str)
    if sensor == "MODIS":
        out["confidence_class"] = clase_modis(df["confidence"].astype(int))
    else:
        out["confidence_class"] = df["confidence"].map(CLASE_VIIRS)
    out["processing_version"] = df["version"].astype(str)
    out["temp_ir_medio_K"] = df[temp_medio]
    out["temp_ir_termico_K"] = df[temp_termico]
    out["scan_km"] = df["scan"]
    out["track_km"] = df["track"]
    out["frp_mw"] = df["frp"]
    out["daynight"] = df["daynight"]
    out["type"] = df["type"].astype("int8")
    out["source_file"] = df["archivo_origen"]
    # filtros como columnas (no se elimina nada)
    out["es_tipo0"] = out["type"] == 0
    out["es_confianza_ok"] = out["confidence_class"] != "baja"
    out["en_filtro_principal"] = out["es_tipo0"] & out["es_confianza_ok"]
    out["frp_negativo"] = out["frp_mw"] < 0
    return out

## 2. Leer, estandarizar y apilar
`version` y `confidence` se leen como texto para no perder el formato original (`6.03`, `61.03`, `l`/`n`/`h`).

In [3]:
tipos = {"acq_time": str, "version": str, "confidence": str}
partes = []

d = pd.read_csv(ENTRADAS["MODIS"], dtype=tipos)
partes.append(estandarizar(d, "MODIS", "brightness", "bright_t31", "modis"))
n_modis = len(d)

d = pd.read_csv(ENTRADAS["VIIRS-SNPP"], dtype=tipos)
partes.append(estandarizar(d, "VIIRS", "bright_ti4", "bright_ti5", "snpp"))
n_snpp = len(d)

d = pd.read_csv(ENTRADAS["VIIRS-NOAA20"], dtype=tipos)
partes.append(estandarizar(d, "VIIRS", "bright_ti4", "bright_ti5", "noaa20"))
n_noaa = len(d)
del d

maestra = pd.concat(partes, ignore_index=True)
del partes
assert len(maestra) == n_modis + n_snpp + n_noaa
print(f"{len(maestra):,} filas = {n_modis:,} + {n_snpp:,} + {n_noaa:,}")
maestra.dtypes

8,606,708 filas = 1,837,958 + 3,847,525 + 2,921,225


latitude                       float64
longitude                      float64
acq_datetime_utc        datetime64[ns]
acq_datetime_bo         datetime64[ns]
acq_date_bo             datetime64[ns]
acq_hour_bo                       int8
platform                        object
sensor                          object
source                          object
resolucion_nominal_m             int64
confidence_original             object
confidence_class                object
processing_version              object
temp_ir_medio_K                float64
temp_ir_termico_K              float64
scan_km                        float64
track_km                       float64
frp_mw                         float64
daynight                        object
type                              int8
source_file                     object
es_tipo0                          bool
es_confianza_ok                   bool
en_filtro_principal               bool
frp_negativo                      bool
dtype: object

## 3. Chequeos

In [4]:
print(maestra["source"].value_counts())
print()
print("Nulos:", maestra.isna().sum()[lambda s: s > 0].to_dict())
print()
print(pd.crosstab(maestra["source"], maestra["confidence_class"]))
print()
print(pd.crosstab(maestra["source"], maestra["type"]))
print()
print("Filas con FRP negativo:", int(maestra["frp_negativo"].sum()))
print("Filas con FRP == 0 (se conservan, sin investigar):", int((maestra["frp_mw"] == 0).sum()))
print()
print("Version de procesamiento por año (UTC):")
print(pd.crosstab(maestra["acq_datetime_utc"].dt.year, [maestra["sensor"], maestra["processing_version"]]))

source
VIIRS-Suomi-NPP    3847525
VIIRS-NOAA-20      2921225
MODIS-Aqua         1111992
MODIS-Terra         725966
Name: count, dtype: int64



Nulos: {}



confidence_class    alta    baja  nominal
source                                   
MODIS-Aqua        402917   50919   658156
MODIS-Terra       269710   43734   412522
VIIRS-NOAA-20     147548  175913  2597764
VIIRS-Suomi-NPP   236253  228524  3382748



type                   0   1     2     3
source                                  
MODIS-Aqua       1109782  15  2195     0
MODIS-Terra       723325  18  2623     0
VIIRS-NOAA-20    2914369   0  3332  3524
VIIRS-Suomi-NPP  3837979   0  5585  3961

Filas con FRP negativo: 16
Filas con FRP == 0 (se conservan, sin investigar): 1216

Version de procesamiento por año (UTC):


sensor              MODIS                    VIIRS
processing_version   6.03     6.2   61.03        2
acq_datetime_utc                                  
2000                    0     869       0        0
2001                    0   14132       0        0
2002                    0   65502       0        0
2003                    0   53733       0        0
2004                    0  103384       0        0
2005                    0   88897       0        0
2006                    0   71165       0        0
2007                    0  105304       0        0
2008                    0   71340       0        0
2009                    0   43297       0        0
2010                    0  151119       0        0
2011                    0   63326       0        0
2012                    0   41706       0   159400
2013                    0   29742       0   115772
2014                    0   28756       0   101746
2015                    0   41002       0   146577
2016                    0   683

## 4. Cobertura por fuente y año
Marca los años con cobertura incompleta (primer y último día con datos, y fracción del año cubierta). Se usa en las series de tiempo para no comparar años parciales con completos.

In [5]:
maestra["anio_utc"] = maestra["acq_datetime_utc"].dt.year.astype("int16")
fechas = maestra["acq_datetime_utc"].dt.normalize()
cob = (maestra.assign(fecha=fechas)
       .groupby(["source", "anio_utc"])["fecha"].agg(primer_dia="min", ultimo_dia="max").reset_index())
cob["dias_cubiertos"] = (cob["ultimo_dia"] - cob["primer_dia"]).dt.days + 1
cob["dias_anio"] = cob["anio_utc"].map(lambda a: 366 if (a % 4 == 0 and (a % 100 != 0 or a % 400 == 0)) else 365)
cob["fraccion_anio"] = (cob["dias_cubiertos"] / cob["dias_anio"]).round(3)
cob["cobertura_parcial"] = cob["fraccion_anio"] < 0.98
cob.to_csv(SALIDA / "cobertura_por_fuente_anio.csv", index=False)
cob[cob["cobertura_parcial"]]

,source,anio_utc,primer_dia,ultimo_dia,dias_cubiertos,dias_anio,fraccion_anio,cobertura_parcial
0,MODIS-Aqua,2002,2002-07-04,2002-12-31,181,365,0.496,True
24,MODIS-Terra,2000,2000-11-01,2000-12-31,61,366,0.167,True
50,VIIRS-NOAA-20,2018,2018-04-01,2018-12-31,275,365,0.753,True
58,VIIRS-Suomi-NPP,2012,2012-01-20,2012-12-31,347,366,0.948,True


## 5. Guardar
Parquet: tabla maestra completa (todas las filas) y copia particionada por `sensor` y `anio_utc` para leer solo lo necesario.

In [6]:
maestra.to_parquet(SALIDA / "firms_bolivia_armonizado.parquet", index=False)
maestra.to_parquet(SALIDA / "firms_bolivia_particionado", partition_cols=["sensor", "anio_utc"], index=False)
for p in sorted(SALIDA.glob("*.parquet")):
    print(p.name, round(p.stat().st_size / 1e6, 1), "MB")

firms_bolivia_armonizado.parquet 181.3 MB


## 6. Diccionario de la tabla maestra
| Columna | Descripción |
|---|---|
| `latitude`, `longitude` | Centro del píxel de fuego |
| `acq_datetime_utc` | Fecha-hora de la pasada, UTC (original) |
| `acq_datetime_bo`, `acq_date_bo`, `acq_hour_bo` | Misma pasada en hora boliviana (UTC−4) |
| `platform` | Terra, Aqua, Suomi-NPP, NOAA-20 |
| `sensor` | MODIS o VIIRS |
| `source` | `sensor-plataforma` (identifica cada serie) |
| `resolucion_nominal_m` | 1000 (MODIS) o 375 (VIIRS) |
| `confidence_original` | Valor original (0–100 o l/n/h) |
| `confidence_class` | baja / nominal / alta |
| `processing_version` | Texto original de `version`. Coincide con el periodo: MODIS 6.2 (2000–2017), 6.03 (2018–2022), 61.03 (2023–2025). VIIRS: 2 |
| `temp_ir_medio_K` | `brightness` (MODIS) / `bright_ti4` (VIIRS) |
| `temp_ir_termico_K` | `bright_t31` (MODIS) / `bright_ti5` (VIIRS) |
| `scan_km`, `track_km` | Tamaño real del píxel |
| `frp_mw` | Potencia radiativa del fuego (MW) |
| `daynight` | D / N |
| `type` | 0 vegetación, 1 volcán, 2 fuente estática, 3 costa/agua |
| `es_tipo0`, `es_confianza_ok`, `en_filtro_principal` | Banderas de filtrado (tipo 0 y confianza distinta de baja) |
| `frp_negativo` | FRP < 0 (se conserva, solo se marca) |
| `source_file` | Archivo anual de origen |
| `anio_utc` | Año de la pasada en UTC |